# SE3082 Lab 09 — Introduction to CUDA

Completed Google Colab notebook. Select a GPU runtime under **Runtime → Change runtime type**, then run the cells in order. Rename this notebook with your student ID before submitting. The output cells are blank until you run it on a CUDA GPU.


## Exercise 1 — Set up CUDA

The first cell reports the GPU and the CUDA compiler. The architecture is detected from the GPU's compute capability so the compilation cells work with a T4, L4, A100, or H100.


In [ ]:
import shutil, subprocess
subprocess.run(['nvidia-smi'], check=True)
NVCC = shutil.which('nvcc') or '/usr/local/cuda/bin/nvcc'
subprocess.run([NVCC, '--version'], check=True)
try:
    capability = subprocess.check_output(
        ['nvidia-smi', '--query-gpu=compute_cap', '--format=csv,noheader'],
        text=True, stderr=subprocess.DEVNULL
    ).splitlines()[0].strip()
except (subprocess.CalledProcessError, IndexError):
    gpu_name = subprocess.check_output(
        ['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], text=True
    ).splitlines()[0]
    capability = next((cap for name, cap in [('T4', '7.5'), ('L4', '8.9'),
                       ('A100', '8.0'), ('H100', '9.0')] if name in gpu_name), None)
    if capability is None:
        raise RuntimeError(f'Unknown GPU {gpu_name}: set ARCH manually')
ARCH = 'sm_' + capability.replace('.', '')
print(f'Compute capability: {capability}; compiler architecture: {ARCH}')


## Exercise 1 — GPU smoke test

Expected: four greeting lines, one per GPU thread; their order can vary.


In [ ]:
%%writefile hello.cu
#include <cstdio>
#include <cuda_runtime.h>

__global__ void hello() {
    printf("Hello from GPU thread %d\n", threadIdx.x);
}

int main() {
    hello<<<1, 4>>>();
    cudaError_t err = cudaGetLastError();
    if (err == cudaSuccess) err = cudaDeviceSynchronize();
    if (err != cudaSuccess) {
        fprintf(stderr, "CUDA error: %s\n", cudaGetErrorString(err));
        return 1;
    }
    return 0;
}


In [ ]:
import subprocess
subprocess.run([NVCC, f'-arch={ARCH}', 'hello.cu', '-o', 'hello'], check=True)
subprocess.run(['./hello'], check=True)


## Exercise 2 — Add two integers

Expected: `Result is 9`.


In [ ]:
%%writefile ex2_add.cu
#include <cstdio>
#include <cuda_runtime.h>

#define CUDA_CHECK(call) do { \
    cudaError_t err = (call); \
    if (err != cudaSuccess) { \
        fprintf(stderr, "%s failed: %s\n", #call, cudaGetErrorString(err)); \
        return 1; \
    } \
} while (0)

__global__ void add(const int *a, const int *b, int *c) {
    *c = *a + *b;
}

int main() {
    int a = 2, b = 7, c = 0;
    int *d_a, *d_b, *d_c;
    CUDA_CHECK(cudaMalloc((void **)&d_a, sizeof(int)));
    CUDA_CHECK(cudaMalloc((void **)&d_b, sizeof(int)));
    CUDA_CHECK(cudaMalloc((void **)&d_c, sizeof(int)));
    CUDA_CHECK(cudaMemcpy(d_a, &a, sizeof(int), cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_b, &b, sizeof(int), cudaMemcpyHostToDevice));
    add<<<1, 1>>>(d_a, d_b, d_c);
    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaMemcpy(&c, d_c, sizeof(int), cudaMemcpyDeviceToHost));
    printf("Result is %d\n", c);
    CUDA_CHECK(cudaFree(d_a));
    CUDA_CHECK(cudaFree(d_b));
    CUDA_CHECK(cudaFree(d_c));
    return c == 9 ? 0 : 1;
}


In [ ]:
import subprocess
subprocess.run([NVCC, f'-arch={ARCH}', 'ex2_add.cu', '-o', 'ex2_add'], check=True)
subprocess.run(['./ex2_add'], check=True)


**Think answers:** Host variables `a`, `b`, and `c` live in CPU memory; the kernel uses device addresses, held in `d_a`, `d_b`, and `d_c`. `&a` gives the address of the host integer to copy, whereas `d_a` already contains a device address. The launch `<<<1, 1>>>` runs one thread. `<<<1, 10>>>` would run ten, but all ten would write the same `c` location, creating a data race even though the arithmetic result is the same.


## Exercise 3 — 512 blocks, one thread each

Expected: 512 result lines and `Verification: 0 mismatches out of 512`.


In [ ]:
%%writefile ex3_blocks.cu
#include <cstdio>
#include <cstdlib>
#include <cuda_runtime.h>

#define N 512
#define CUDA_CHECK(call) do { \
    cudaError_t err = (call); \
    if (err != cudaSuccess) { \
        fprintf(stderr, "%s failed: %s\n", #call, cudaGetErrorString(err)); \
        return 1; \
    } \
} while (0)

__global__ void add(const int *a, const int *b, int *c) {
    int i = blockIdx.x;
    c[i] = a[i] + b[i];
}

void random_ints(int *x, int count) {
    for (int i = 0; i < count; ++i) x[i] = rand() % 100;
}

int main() {
    const size_t bytes = (size_t)N * sizeof(int);
    int *a = (int *)malloc(bytes), *b = (int *)malloc(bytes), *c = (int *)malloc(bytes);
    if (!a || !b || !c) { fprintf(stderr, "Host allocation failed\n"); return 1; }
    random_ints(a, N);
    random_ints(b, N);

    int *d_a, *d_b, *d_c;
    CUDA_CHECK(cudaMalloc((void **)&d_a, bytes));
    CUDA_CHECK(cudaMalloc((void **)&d_b, bytes));
    CUDA_CHECK(cudaMalloc((void **)&d_c, bytes));
    CUDA_CHECK(cudaMemcpy(d_a, a, bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_b, b, bytes, cudaMemcpyHostToDevice));
    add<<<N, 1>>>(d_a, d_b, d_c);
    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaMemcpy(c, d_c, bytes, cudaMemcpyDeviceToHost));

    int errors = 0;
    for (int i = 0; i < N; ++i) {
        printf("%d + %d = %d\n", a[i], b[i], c[i]);
        if (c[i] != a[i] + b[i]) ++errors;
    }
    printf("Verification: %d mismatches out of %d\n", errors, N);
    free(a); free(b); free(c);
    CUDA_CHECK(cudaFree(d_a));
    CUDA_CHECK(cudaFree(d_b));
    CUDA_CHECK(cudaFree(d_c));
    return errors ? 1 : 0;
}


In [ ]:
import subprocess
subprocess.run([NVCC, f'-arch={ARCH}', 'ex3_blocks.cu', '-o', 'ex3_blocks'], check=True)
subprocess.run(['./ex3_blocks'], check=True)


**Prediction:** `<<<512, 1>>>` launches 512 blocks and 512 threads. A block with one thread still occupies a warp with 32 lanes, leaving 31 lanes inactive.


## Exercise 4 — One block, 512 threads

Expected: 512 result lines and `Verification: 0 mismatches out of 512`.


In [ ]:
%%writefile ex4_threads.cu
#include <cstdio>
#include <cstdlib>
#include <cuda_runtime.h>

#ifndef N
#define N 512
#endif
#define CUDA_CHECK(call) do { \
    cudaError_t err = (call); \
    if (err != cudaSuccess) { \
        fprintf(stderr, "%s failed: %s\n", #call, cudaGetErrorString(err)); \
        return 1; \
    } \
} while (0)

__global__ void addT(const int *a, const int *b, int *c) {
    int i = threadIdx.x;
    c[i] = a[i] + b[i];
}

void random_ints(int *x, int count) {
    for (int i = 0; i < count; ++i) x[i] = rand() % 100;
}

int main() {
    const size_t bytes = (size_t)N * sizeof(int);
    int *a = (int *)malloc(bytes), *b = (int *)malloc(bytes), *c = (int *)malloc(bytes);
    if (!a || !b || !c) { fprintf(stderr, "Host allocation failed\n"); return 1; }
    random_ints(a, N);
    random_ints(b, N);

    int *d_a, *d_b, *d_c;
    CUDA_CHECK(cudaMalloc((void **)&d_a, bytes));
    CUDA_CHECK(cudaMalloc((void **)&d_b, bytes));
    CUDA_CHECK(cudaMalloc((void **)&d_c, bytes));
    CUDA_CHECK(cudaMemcpy(d_a, a, bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_b, b, bytes, cudaMemcpyHostToDevice));
    addT<<<1, N>>>(d_a, d_b, d_c);
    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaMemcpy(c, d_c, bytes, cudaMemcpyDeviceToHost));

    int errors = 0;
    for (int i = 0; i < N; ++i) {
        printf("%d) %d + %d = %d\n", i, a[i], b[i], c[i]);
        if (c[i] != a[i] + b[i]) ++errors;
    }
    printf("Verification: %d mismatches out of %d\n", errors, N);
    free(a); free(b); free(c);
    CUDA_CHECK(cudaFree(d_a));
    CUDA_CHECK(cudaFree(d_b));
    CUDA_CHECK(cudaFree(d_c));
    return errors ? 1 : 0;
}


In [ ]:
import subprocess
subprocess.run([NVCC, f'-arch={ARCH}', 'ex4_threads.cu', '-o', 'ex4_threads'], check=True)
subprocess.run(['./ex4_threads'], check=True)


### Exercise 4 experiment — 2,048 threads in one block

The next cell recompiles the same source with `N=2048`. Its launch is expected to fail with `invalid configuration argument` because 2,048 exceeds the device's maximum threads per block (normally 1,024). The `cudaGetLastError` check reports it. The source retains its default `N=512`; the final cell recompiles that version.


In [ ]:
subprocess.run([NVCC, f'-arch={ARCH}', '-DN=2048', 'ex4_threads.cu', '-o', 'ex4_limit'], check=True)
limit_run = subprocess.run(['./ex4_limit'])
assert limit_run.returncode != 0, 'Expected the 2,048-thread launch to fail'
print('Expected launch-configuration failure observed.')
subprocess.run([NVCC, f'-arch={ARCH}', 'ex4_threads.cu', '-o', 'ex4_threads'], check=True)


## Exercise 5 — Multiply 10,000,000 vector elements

Expected: `Launching 19532 blocks x 512 threads`, 1,000 final results, and `Verification: 0 mismatches out of 10000000`.


In [ ]:
%%writefile ex5_vecmul.cu
#include <cstdio>
#include <cstdlib>
#include <cuda_runtime.h>

#define N 10000000
#define THREADS_PER_BLOCK 512
#define CUDA_CHECK(call) do { \
    cudaError_t err = (call); \
    if (err != cudaSuccess) { \
        fprintf(stderr, "%s failed: %s\n", #call, cudaGetErrorString(err)); \
        return 1; \
    } \
} while (0)

__global__ void vecMul(const int *a, const int *b, int *c, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) c[i] = a[i] * b[i];
}

void random_ints(int *x, int count) {
    for (int i = 0; i < count; ++i) x[i] = rand() % 100;
}

int main() {
    const size_t bytes = (size_t)N * sizeof(int);
    int *a = (int *)malloc(bytes), *b = (int *)malloc(bytes), *c = (int *)malloc(bytes);
    if (!a || !b || !c) { fprintf(stderr, "Host allocation failed\n"); return 1; }
    random_ints(a, N);
    random_ints(b, N);

    int *d_a, *d_b, *d_c;
    CUDA_CHECK(cudaMalloc((void **)&d_a, bytes));
    CUDA_CHECK(cudaMalloc((void **)&d_b, bytes));
    CUDA_CHECK(cudaMalloc((void **)&d_c, bytes));
    CUDA_CHECK(cudaMemcpy(d_a, a, bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_b, b, bytes, cudaMemcpyHostToDevice));

    int blocks = (N + THREADS_PER_BLOCK - 1) / THREADS_PER_BLOCK;
    printf("Launching %d blocks x %d threads\n", blocks, THREADS_PER_BLOCK);
    vecMul<<<blocks, THREADS_PER_BLOCK>>>(d_a, d_b, d_c, N);
    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaMemcpy(c, d_c, bytes, cudaMemcpyDeviceToHost));

    for (int i = N - 1000; i < N; ++i)
        printf("%d) %d x %d = %d\n", i, a[i], b[i], c[i]);

    int errors = 0;
    for (int i = 0; i < N; ++i)
        if (c[i] != a[i] * b[i]) ++errors;
    printf("Verification: %d mismatches out of %d\n", errors, N);

    free(a); free(b); free(c);
    CUDA_CHECK(cudaFree(d_a));
    CUDA_CHECK(cudaFree(d_b));
    CUDA_CHECK(cudaFree(d_c));
    return errors ? 1 : 0;
}


In [ ]:
import subprocess
subprocess.run([NVCC, f'-arch={ARCH}', 'ex5_vecmul.cu', '-o', 'ex5_vecmul'], check=True)
subprocess.run(['./ex5_vecmul'], check=True)


**Think answers:** 512 is a multiple of the warp width, 32, so all lanes in each full block can do useful work. With 500 threads per block, ceiling division gives 20,000 blocks and exactly 10,000,000 launched threads; each block's last warp has only 20 active lanes. The bounds check is still needed for the general case.


## Exercise 6 — Multiply 10,000 × 10,000 matrix elements

This is element-wise multiplication. It uses three flat arrays of 400 MB on the host and three on the device. Expected: `Grid: 313 x 625 blocks`, the final 1,000 elements of row 9999, and `Verification: 0 mismatches out of 100000000`.


In [ ]:
%%writefile ex6_matrix.cu
#include <cstdio>
#include <cstdlib>
#include <cuda_runtime.h>

#define ROWS 10000
#define COLS 10000
#define CUDA_CHECK(call) do { \
    cudaError_t err = (call); \
    if (err != cudaSuccess) { \
        fprintf(stderr, "%s failed: %s\n", #call, cudaGetErrorString(err)); \
        return 1; \
    } \
} while (0)

// Element-wise product, with x selecting a column and y selecting a row.
__global__ void mulElem(const int *a, const int *b, int *c, int rows, int cols) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    if (row < rows && col < cols) {
        size_t idx = (size_t)row * cols + col;
        c[idx] = a[idx] * b[idx];
    }
}

void random_ints(int *x, size_t count) {
    for (size_t i = 0; i < count; ++i) x[i] = rand() % 100;
}

int main() {
    const size_t count = (size_t)ROWS * COLS;
    const size_t bytes = count * sizeof(int);
    int *a = (int *)malloc(bytes), *b = (int *)malloc(bytes), *c = (int *)malloc(bytes);
    if (!a || !b || !c) { fprintf(stderr, "Host allocation failed\n"); return 1; }
    random_ints(a, count);
    random_ints(b, count);

    int *d_a, *d_b, *d_c;
    CUDA_CHECK(cudaMalloc((void **)&d_a, bytes));
    CUDA_CHECK(cudaMalloc((void **)&d_b, bytes));
    CUDA_CHECK(cudaMalloc((void **)&d_c, bytes));
    CUDA_CHECK(cudaMemcpy(d_a, a, bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_b, b, bytes, cudaMemcpyHostToDevice));

    dim3 threadsPerBlock(32, 16);
    dim3 numBlocks((COLS + threadsPerBlock.x - 1) / threadsPerBlock.x,
                   (ROWS + threadsPerBlock.y - 1) / threadsPerBlock.y);
    printf("Grid: %u x %u blocks\n", numBlocks.x, numBlocks.y);
    mulElem<<<numBlocks, threadsPerBlock>>>(d_a, d_b, d_c, ROWS, COLS);
    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaMemcpy(c, d_c, bytes, cudaMemcpyDeviceToHost));

    for (int col = COLS - 1000; col < COLS; ++col) {
        size_t idx = (size_t)(ROWS - 1) * COLS + col;
        printf("C[%d][%d]: %d x %d = %d\n", ROWS - 1, col, a[idx], b[idx], c[idx]);
    }

    size_t errors = 0;
    for (size_t i = 0; i < count; ++i)
        if (c[i] != a[i] * b[i]) ++errors;
    printf("Verification: %zu mismatches out of %zu\n", errors, count);

    free(a); free(b); free(c);
    CUDA_CHECK(cudaFree(d_a));
    CUDA_CHECK(cudaFree(d_b));
    CUDA_CHECK(cudaFree(d_c));
    return errors ? 1 : 0;
}


In [ ]:
import subprocess
subprocess.run([NVCC, f'-arch={ARCH}', 'ex6_matrix.cu', '-o', 'ex6_matrix'], check=True)
subprocess.run(['./ex6_matrix'], check=True)


## Finish

Check the verification lines and GPU output, save the notebook, and submit it as your lecturer instructs. Release the GPU using **Runtime → Disconnect and delete runtime**.
